# Silver — Vehicle Registry (SCD Type 2)

Cleans and historises the ZTM vehicle registry.

**Source:** `bronze_vehicles` (ZTM vehicle database, batch)
**Target:** `silver_vehicles_scd` — one row per *version* of a vehicle
**Quarantine:** `silver_vehicles_scd_quarantine` — rejected rows with their reasons

Vehicles change slowly (refurbishment adds air conditioning or USB, seating
capacity changes, a vehicle moves between operators), so the table keeps full
history via `valid_from` / `valid_to` / `is_current`. Changes are detected by
comparing a hash of the tracked attributes, which also makes the load
idempotent.

Re-running it doesn't produce new versions.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

dbutils.widgets.text("catalog","")
dbutils.widgets.text("schema","")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

bronze_vehicles     = f"{catalog}.{schema}.bronze_vehicles"
silver_vehicles     = f"{catalog}.{schema}.silver_vehicles_scd"
vehicles_quarantine = f"{catalog}.{schema}.silver_vehicles_quarantine_scd"

# One timestamp for the whole run so valid_to of the old version exactly
# matches valid_from of the new one. No gaps and no overlaps.
batch_ts = datetime.now()

print(f"Source:     {bronze_vehicles}")
print(f"Target:     {silver_vehicles}")
print(f"Quarantine: {vehicles_quarantine}")
print(f"Batch timestamp: {batch_ts}")

In [0]:
from pyspark.sql import Window

bronze = spark.read.table(bronze_vehicles)

# Cleaning: fix types, trim strings, repair the source's escaped quotes
cleaned = (bronze
    .withColumn("vehicleCode",     F.trim(F.col("vehicleCode")))
    .withColumn("model",           F.regexp_replace(F.trim(F.col("model")), '^"+|"+$', ""))
    .withColumn("model",           F.regexp_replace(F.col("model"), '""', '"'))
    .withColumn("carrier",         F.trim(F.col("carrirer")))
    .withColumn("productionYear",  F.col("productionYear").cast("int"))
    .withColumn("seats",           F.col("seats").cast("int"))
    .withColumn("standingPlaces",  F.col("standingPlaces").cast("int"))
    .withColumn("length",          F.col("length").cast("decimal(5,2)"))
    .withColumn("passengersDoors", F.col("passengersDoors").cast("int"))
    .withColumn("bikeHolders",     F.col("bikeHolders").cast("int"))
)


# Deduplication: Bronze is "append-only", so a vehicle can appear several times.
# Keeping only the most recently ingested record per business key.
w = Window.partitionBy("vehicleCode").orderBy(F.col("ingestion_timestamp").desc())
cleaned = (cleaned
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn"))

# Explicit, named quality rules (each one produces a rejection reason)
rules = {
    "missing_vehicle_code":    F.col("vehicleCode").isNull() | (F.col("vehicleCode") == ""),
    "invalid_transport_type": ~F.col("transportationType").isin("Tramwaj", "Autobus"),
    "negative_seats":          F.col("seats") < 0,
    "negative_standing":       F.col("standingPlaces") < 0,
    "implausible_year":       (F.col("productionYear") < 1900) | (F.col("productionYear") > F.year(F.current_date()) + 1),
    "zero_capacity":          (F.coalesce(F.col("seats"), F.lit(0)) + F.coalesce(F.col("standingPlaces"), F.lit(0))) == 0,
}

# Collect the names of every rule a row violates. 
# array_compact drops the NULLs left by rules that passed, so an empty array means the row is clean.
reasons = F.array_compact(F.array(*[
    F.when(cond, F.lit(name)) for name, cond in rules.items()
]))

evaluated = cleaned.withColumn("rejection_reasons", reasons)

valid    = evaluated.filter(F.size("rejection_reasons") == 0)
rejected = evaluated.filter(F.size("rejection_reasons") > 0)

valid_n, rejected_n = valid.count(), rejected.count()
print(f"Valid:    {valid_n:,}")
print(f"Rejected: {rejected_n:,}")

# Route rejected rows to quarantine, with the reason preserved
if rejected_n > 0:
    (rejected
        .select(
            F.col("vehicleCode"),
            F.to_json(F.struct([bronze[c] for c in bronze.columns])).alias("raw_record"),
            F.col("rejection_reasons"),
            F.col("source_file"),
            F.lit(batch_ts).cast("timestamp").alias("quarantined_at"))
        .write.format("delta").mode("append").saveAsTable(vehicles_quarantine))
    print("Rejected rows written to quarantine.")
    display(rejected.select("vehicleCode", "rejection_reasons").limit(10))

In [0]:
tracked = ["transportationType", "brand", "model", "productionYear", "carrier", "driveType",
           "seats", "standingPlaces", "length", "passengersDoors", "vehicleCharacteristics",
           "floorHeight", "bidirectional", "historicVehicle", "wheelchairsRamp",
           "kneelingMechanism", "airConditioning", "usb", "bikeHolders",
           "voiceAnnouncements", "monitoring", "internalMonitor", "aed", "ticketMachine"]

# A single hash over the tracked attributes replaces comparing 24 columns one by
# one, avoids NULL-comparison pitfalls.
# COALESCE: concat_ws skips NULLs so different records could hash identically.
incoming = (valid
    .withColumn("attributes_hash", F.sha2(F.concat_ws("||",
        *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in tracked]
    ), 256))
    .select("vehicleCode", *tracked, "attributes_hash", "source_file")
)

silver_dt = DeltaTable.forName(spark, silver_vehicles)
current   = spark.read.table(silver_vehicles).filter("is_current = true")

changed = (incoming.alias("s")
    .join(current.alias("t"), "vehicleCode")
    .where(F.col("s.attributes_hash") != F.col("t.attributes_hash"))
    .select("s.*"))

# Changed rows appear twice in the staged source, because a
# single MERGE can only UPDATE or INSERT a given matched row.
#   merge_key = NULL         -> NULL never equals anything, no match -> INSERT the new version
#   merge_key = vehicleCode  -> matches the current row              -> UPDATE closes it
staged = (changed.withColumn("merge_key", F.lit(None).cast("string"))
    .unionByName(incoming.withColumn("merge_key", F.col("vehicleCode"))))

insert_values = {c: f"s.{c}" for c in ["vehicleCode", *tracked, "attributes_hash", "source_file"]}
insert_values.update({
    "valid_from":           F.lit(batch_ts).cast("timestamp"),
    "valid_to":             F.lit(None).cast("timestamp"),
    "is_current":           F.lit(True),
    "silver_ingestion_ts":  F.lit(batch_ts).cast("timestamp"),
})


# Three cases handled by this single MERGE:
#   new vehicle      -> no match                     -> INSERT first version
#   changed vehicle  -> match, attributes changed,   -> UPDATE closes the old version and the merge_key=NULL copy INSERTs the new one
#   unchanged vehicle -> match, attributes identical -> nothing happens
(silver_dt.alias("t")
    .merge(staged.alias("s"), "t.vehicleCode = s.merge_key AND t.is_current = true")
    .whenMatchedUpdate(
        condition = "t.attributes_hash <> s.attributes_hash",
        set = {"valid_to": F.lit(batch_ts).cast("timestamp"), "is_current": F.lit(False)})
    .whenNotMatchedInsert(values = insert_values)
    .execute())


# In SCD2 the row count exceeds the number of vehicles: each historical version is its own row. 
# "Distinct codes" should stay constant across runs, while
# "Historical" grows only when an attribute actually changes.
silver = spark.read.table(silver_vehicles)
print(f"Total rows:     {silver.count():,}")
print(f"Current:        {silver.filter('is_current = true').count():,}")
print(f"Historical:     {silver.filter('is_current = false').count():,}")
print(f"Distinct codes: {silver.select('vehicleCode').distinct().count():,}")

In [0]:
# Simulate a source update refurbished vehicles gain amenities
# In reality this is what a refreshed vehicle registry would look like after three trams were modernised.
# Its not a part of the regular pipeline.
updates = (spark.read.table(bronze_vehicles)
    .filter(F.col("vehicleCode").isin("1001", "1002", "1003"))
    .withColumn("airConditioning", F.lit(True))
    .withColumn("usb", F.lit(True))
    .withColumn("seats", F.col("seats") - 4)     
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

# Simulate broken records arriving from the source
bad = (spark.read.table(bronze_vehicles).limit(3)
    .withColumn("vehicleCode",         F.lit("BAD_001"))
    .withColumn("transportationType",  F.lit("Rakieta"))       # not in allowed values
    .withColumn("seats",               F.lit("-5"))            # negative
    .withColumn("productionYear",      F.lit("1850"))          # implausible
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .limit(1)
)

updates.unionByName(bad).write.format("delta").mode("append").saveAsTable(bronze_vehicles)
print(f"Bronze now has {spark.read.table(bronze_vehicles).count():,} rows.")

In [0]:
display(spark.read.table(silver_vehicles)
    .filter(F.col("vehicleCode") == "1001")
    .select("vehicleCode", "model", "seats", "airConditioning", "usb",
            "valid_from", "valid_to", "is_current")
    .orderBy("valid_from"))